In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import chi2, SelectKBest, RFE
from sklearn.metrics import accuracy_score, classification_report

In [41]:
df = pd.read_csv("wbc.csv")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [42]:
# M = Malignant = 1
# B = Benign    = 0

y = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

In [34]:
num_cols = [
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",

    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave points_se",
    "symmetry_se",
    "fractal_dimension_se",

    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

# Only use numerical variables
X = df[num_cols]

In [35]:
num_tf = Pipeline([
    ("scaler", StandardScaler())
])

In [43]:
from sklearn.feature_selection import f_classif
selector_filter = SelectKBest(
    score_func=f_classif,
    k=5
)

In [37]:
pipe_filter = Pipeline([
    ("prep", num_tf),
    ("sel", selector_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

pipe_filter.fit(X_train, y_train)

pred = pipe_filter.predict(X_test)

print("=== Filter (ANOVA) + LR ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

=== Filter (ANOVA) + LR ===
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [45]:
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()

print("Feature names:")
print(feat_names)
print("\n")


Feature names:
['radius_mean' 'texture_mean' 'perimeter_mean' 'area_mean'
 'smoothness_mean' 'compactness_mean' 'concavity_mean'
 'concave points_mean' 'symmetry_mean' 'fractal_dimension_mean'
 'radius_se' 'texture_se' 'perimeter_se' 'area_se' 'smoothness_se'
 'compactness_se' 'concavity_se' 'concave points_se' 'symmetry_se'
 'fractal_dimension_se' 'radius_worst' 'texture_worst' 'perimeter_worst'
 'area_worst' 'smoothness_worst' 'compactness_worst' 'concavity_worst'
 'concave points_worst' 'symmetry_worst' 'fractal_dimension_worst']




In [46]:
sel = pipe_filter.named_steps["sel"]

mask = sel.get_support()

selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]

top = sorted(
    zip(selected_names, selected_scores),
    key=lambda t: t[1],
    reverse=True
)

print("Top features:")

for feature, score in top:
    print(feature, ":", score)

Top features:
concave points_worst : 733.7249325739482
perimeter_worst : 717.2464871041376
radius_worst : 692.861395260564
concave points_mean : 684.5268452011383
perimeter_mean : 548.4132358502825


In [47]:
results = []

for k in range(1, len(num_cols) + 1):

    selector_filter = SelectKBest(
        score_func=f_classif,
        k=k
    )

    pipe_filter = Pipeline([
        ("prep", num_tf),
        ("sel", selector_filter),
        ("clf", LogisticRegression(max_iter=1000))
    ])

    pipe_filter.fit(X_train, y_train)

    pred = pipe_filter.predict(X_test)

    accuracy = accuracy_score(y_test, pred)

    results.append({
        "Jumlah Fitur": k,
        "Accuracy": accuracy
    })


results_df = pd.DataFrame(results)

display(results_df)

,Jumlah Fitur,Accuracy
0,1,0.929825
1,2,0.956140
2,3,0.956140
3,4,0.956140
4,5,0.964912
5,6,0.964912
6,7,0.964912
7,8,0.964912
8,9,0.973684
9,10,0.956140


In [48]:
max_accuracy = results_df["Accuracy"].max()

best_results = results_df[
    results_df["Accuracy"] == max_accuracy
]

print("Accuracy tertinggi:", max_accuracy)
print("\nJumlah fitur dengan accuracy tertinggi:")
display(best_results)


# Select the smallest number of features
best_k = int(best_results["Jumlah Fitur"].min())

print("Jumlah fitur optimal:", best_k)

Accuracy tertinggi: 0.9824561403508771

Jumlah fitur dengan accuracy tertinggi:


,Jumlah Fitur,Accuracy
13,14,0.982456
17,18,0.982456
18,19,0.982456
19,20,0.982456


Jumlah fitur optimal: 14


In [49]:
selector_filter = SelectKBest(
    score_func=f_classif,
    k=best_k
)

pipe_filter = Pipeline([
    ("prep", num_tf),
    ("sel", selector_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

pipe_filter.fit(X_train, y_train)

pred = pipe_filter.predict(X_test)

In [50]:
print("\n=== FINAL MODEL ===")
print("Jumlah fitur:", best_k)
print("Accuracy:", accuracy_score(y_test, pred))
print()
print(classification_report(y_test, pred))



=== FINAL MODEL ===
Jumlah fitur: 14
Accuracy: 0.9824561403508771

              precision    recall  f1-score   support

           0       0.97      1.00      0.99        72
           1       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [51]:
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()

sel = pipe_filter.named_steps["sel"]

mask = sel.get_support()

selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]

top = sorted(
    zip(selected_names, selected_scores),
    key=lambda t: t[1],
    reverse=True
)

print("=== FINAL SELECTED FEATURES ===")

for i, (feature, score) in enumerate(top, start=1):
    print(f"{i}. {feature} : {score:.4f}")

=== FINAL SELECTED FEATURES ===
1. concave points_worst : 733.7249
2. perimeter_worst : 717.2465
3. radius_worst : 692.8614
4. concave points_mean : 684.5268
5. perimeter_mean : 548.4132
6. area_worst : 522.1889
7. radius_mean : 511.2748
8. area_mean : 444.8575
9. concavity_mean : 397.5921
10. concavity_worst : 319.5078
11. compactness_mean : 263.5643
12. compactness_worst : 238.2037
13. radius_se : 205.4332
14. perimeter_se : 193.1689
